# Vision Dataset Collection and Preparation

This notebook guides you through collecting and preparing computer vision datasets for KarigAI.

## Datasets to Collect:
1. Equipment images (appliances, machinery, tools)
2. Error code displays
3. Traditional patterns and designs
4. Product quality images (saffron, textiles, crops)
5. Circuit board images
6. Crop disease images

In [ ]:
import sys
sys.path.append('..')

from utils.vision_dataset_preparation import VisionDatasetManager
from utils.vision_augmentation import VisionAugmentation, SyntheticErrorCodeGenerator
import cv2
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

## 1. Initialize Dataset Manager

In [ ]:
# Initialize dataset manager
manager = VisionDatasetManager(base_dir='../data/vision')

# Create directory structures for all dataset types
for dataset_type in manager.dataset_types:
    manager.create_dataset_structure(dataset_type)
    print(f"Created structure for: {dataset_type}")

## 2. Generate Synthetic Error Codes

Since error code displays are hard to collect, we'll generate synthetic ones.

In [ ]:
# Initialize synthetic generator
generator = SyntheticErrorCodeGenerator(image_size=(320, 240))

# Define common error codes
error_codes = [
    'E01', 'E02', 'E03', 'E04', 'E05',
    'F01', 'F02', 'F03', 'F04', 'F05',
    'ERR-001', 'ERR-002', 'ERR-003',
    '0001', '0002', '0003', '0004', '0005'
]

# Generate synthetic images
print("Generating synthetic error codes...")
synthetic_images = generator.generate_batch(error_codes, num_variations=10)
print(f"Generated {len(synthetic_images)} images")

# Display examples
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for i, ax in enumerate(axes.flat):
    if i < len(synthetic_images):
        img, label = synthetic_images[i]
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.set_title(f"Code: {label}")
        ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Save synthetic images
output_dir = Path('../data/vision/error_codes/raw/synthetic')
output_dir.mkdir(parents=True, exist_ok=True)

for i, (img, label) in enumerate(synthetic_images):
    output_path = output_dir / f"synthetic_{label.replace('-', '_')}_{i:04d}.png"
    cv2.imwrite(str(output_path), img)

print(f"Saved {len(synthetic_images)} synthetic images to: {output_dir}")

## 3. Download Public Datasets

### PlantVillage Dataset (Crop Diseases)

In [ ]:
# Download PlantVillage dataset
# Note: This requires manual download or Kaggle API

print("To download PlantVillage dataset:")
print("1. Visit: https://www.kaggle.com/datasets/emmarex/plantdisease")
print("2. Download the dataset")
print("3. Extract to: ml_models/data/vision/crops/raw/plantvillage/")
print("")
print("Or use Kaggle API:")
print("  kaggle datasets download -d emmarex/plantdisease")
print("  unzip plantdisease.zip -d ../data/vision/crops/raw/plantvillage/")

### ImageNet Subset (Equipment)

In [ ]:
print("To download ImageNet equipment images:")
print("1. Visit: https://www.image-net.org/")
print("2. Search for categories: refrigerator, washing_machine, air_conditioner, etc.")
print("3. Download images for each category")
print("4. Place in: ml_models/data/vision/equipment/raw/[category]/")

## 4. Organize Dataset

Once you have collected raw images, organize them into train/val/test splits.

In [ ]:
# Example: Organize error codes dataset
dataset_type = 'error_codes'
source_dir = '../data/vision/error_codes/raw'

# Check if source directory has images
source_path = Path(source_dir)
if source_path.exists():
    image_count = len(list(source_path.rglob('*.png'))) + len(list(source_path.rglob('*.jpg')))
    print(f"Found {image_count} images in {source_dir}")
    
    if image_count > 0:
        # Organize into splits
        manager.organize_dataset(
            dataset_type=dataset_type,
            source_dir=source_dir,
            train_ratio=0.8,
            val_ratio=0.1,
            copy_files=True
        )
    else:
        print("No images found. Please add images to the raw directory first.")
else:
    print(f"Source directory not found: {source_dir}")

## 5. Validate Dataset

In [ ]:
# Validate dataset integrity
results = manager.validate_dataset('error_codes', fix_corrupted=False)

# Display results
for split, info in results.items():
    print(f"\n{split.upper()}:")
    print(f"  Status: {info.get('status', 'N/A')}")
    print(f"  Total: {info.get('total', 0)}")
    print(f"  Valid: {info.get('valid', 0)}")
    print(f"  Corrupted: {info.get('corrupted', 0)}")

## 6. Analyze Dataset Statistics

In [ ]:
# Analyze dataset
analysis = manager.analyze_dataset('error_codes', split='train')

# Visualize statistics
if analysis:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    # Width distribution
    axes[0].bar(['Min', 'Mean', 'Max'], 
                [analysis['width']['min'], analysis['width']['mean'], analysis['width']['max']])
    axes[0].set_title('Image Width')
    axes[0].set_ylabel('Pixels')
    
    # Height distribution
    axes[1].bar(['Min', 'Mean', 'Max'], 
                [analysis['height']['min'], analysis['height']['mean'], analysis['height']['max']])
    axes[1].set_title('Image Height')
    axes[1].set_ylabel('Pixels')
    
    # File size distribution
    axes[2].bar(['Min', 'Mean', 'Max'], 
                [analysis['file_size_kb']['min'], analysis['file_size_kb']['mean'], 
                 analysis['file_size_kb']['max']])
    axes[2].set_title('File Size')
    axes[2].set_ylabel('KB')
    
    plt.tight_layout()
    plt.show()

## 7. Generate Manifest

In [ ]:
# Generate manifest file
manifest = manager.generate_manifest('error_codes')

# Display manifest
import json
print(json.dumps(manifest, indent=2))

## 8. Test Augmentation Pipeline

In [ ]:
# Initialize augmentation
aug = VisionAugmentation(image_size=224)

# Load a sample image
sample_images = list(Path('../data/vision/error_codes/processed/train').glob('*.png'))[:1]

if sample_images:
    img_path = sample_images[0]
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Apply augmentations
    train_transform = aug.get_ocr_train_transforms()
    
    # Show original and augmented versions
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    
    axes[0, 0].imshow(img)
    axes[0, 0].set_title('Original')
    axes[0, 0].axis('off')
    
    for i in range(1, 8):
        augmented = train_transform(image=img)
        aug_img = augmented['image'].permute(1, 2, 0).numpy()
        
        # Denormalize for visualization
        mean = np.array([0.485, 0.456, 0.406])
        std = np.array([0.229, 0.224, 0.225])
        aug_img = std * aug_img + mean
        aug_img = np.clip(aug_img, 0, 1)
        
        row = i // 4
        col = i % 4
        axes[row, col].imshow(aug_img)
        axes[row, col].set_title(f'Augmented {i}')
        axes[row, col].axis('off')
    
    plt.tight_layout()
    plt.show()
else:
    print("No sample images found. Please organize dataset first.")

## 9. Create Sample Dataset for Testing

In [ ]:
# Create small sample dataset for quick testing
manager.create_sample_dataset('error_codes', num_samples=50)

print("Sample dataset created!")
print("Use this for quick model prototyping and testing.")

## 10. Summary and Next Steps

In [ ]:
print("Dataset Collection Summary:")
print("="*50)

for dataset_type in manager.dataset_types:
    dataset_path = Path(f'../data/vision/{dataset_type}/processed')
    if dataset_path.exists():
        train_count = len(list((dataset_path / 'train').glob('*.*')))
        val_count = len(list((dataset_path / 'val').glob('*.*')))
        test_count = len(list((dataset_path / 'test').glob('*.*')))
        total = train_count + val_count + test_count
        
        print(f"\n{dataset_type.upper()}:")
        print(f"  Train: {train_count}")
        print(f"  Val: {val_count}")
        print(f"  Test: {test_count}")
        print(f"  Total: {total}")
        
        if total > 0:
            print(f"  ✓ Ready for training")
        else:
            print(f"  ✗ No images - needs collection")
    else:
        print(f"\n{dataset_type.upper()}: Not organized yet")

print("\n" + "="*50)
print("\nNext Steps:")
print("1. Collect more images for datasets with low counts")
print("2. Annotate images with labels and bounding boxes")
print("3. Proceed to model training (Task 17.2-17.7)")
print("4. Evaluate models on test sets")
print("5. Deploy models to production")